# SWAN AI-SQL on a SWAN 2.0 database

This notebook installs `swan-aisql` from PyPI and runs five questions of the [SWAN 2.0 benchmark](https://github.com/ZhaoFuheng/SWANBench) on its `formula_1` database, the one every system scores highest on (mean quality over its 30 questions, gpt-5.6-luna):

| system | formula_1 | california_schools | european_football_2 | superhero |
|---|---|---|---|---|
| LOTUS | 1.000 | 0.733 | 0.801 | 0.507 |
| BlendSQL | 0.984 | 0.717 | 0.845 | 0.498 |
| Palimpzest | 0.968 | 0.717 | 0.850 | 0.529 |
| PLOP | 0.902 | 0.679 | 0.671 | 0.508 |
| SWAN-AISQL | 0.995 | 0.782 | 0.818 | 0.457 |

Each question is one SQL query with `ai_filter` predicates over plain string expressions. The engine decides the model calls: it folds duplicate prompts, orders predicates by learned selectivity, pushes a `LIMIT` into the calls and keeps twenty requests in flight. `ai_usage()` reports what each query cost.

You need an OpenAI key in `OPENAI_API_KEY`. The five questions below make about 250 model calls, a few cents.

In [1]:
# Once, in this environment. The `embed` extra adds the local embeddings server behind the learned predicate
# ordering (sentence-transformers + torch, about 1.2 GB of models on first start); without it every query still
# runs, ordered by prompt cost alone.
# %pip install "swan-aisql[embed]" pandas

## The database

SWAN 2.0 builds its databases from BIRD's `dev` set: large tables sampled to about ten thousand rows, entities duplicated, and the columns that the questions ask about removed, so that only a model can answer. The benchmark repository builds them with one command; the cell below clones it and runs that command when the database is not already there (`SWAN2_DB` points at an existing copy).

In [2]:
import os, subprocess, sys
from pathlib import Path

DB = Path(os.environ.get("SWAN2_DB", "SWANBench/data/databases/masked/formula_1/formula_1.sqlite"))
if not DB.is_file():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ZhaoFuheng/SWANBench"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./SWANBench"], check=True)
    subprocess.run(["swan-bench", "prepare"], check=True, env={**os.environ, "SWAN_BENCH_ROOT": "SWANBench"})
print(DB, f"{DB.stat().st_size / 1e6:.0f} MB")

SWANBench/data/databases/masked/formula_1/formula_1.sqlite 10 MB


## Connect

`swan_aisql.connect()` opens a DuckDB connection with the extension loaded. With no endpoint given it talks to OpenAI directly using `OPENAI_API_KEY`, model `gpt-5.6-luna`, and starts the embeddings server when nothing answers on `localhost:4002`. The tables are copied in from the SQLite file.

In [3]:
import sqlite3
import pandas as pd
import swan_aisql

con = swan_aisql.connect()
con.execute("SET enable_progress_bar = false")  # no progress widgets in the notebook output

src = sqlite3.connect(DB)
tables = [r[0] for r in src.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]
for t in tables:
    df = pd.read_sql_query(f'SELECT * FROM "{t}"', src)
    con.register("df", df)
    con.execute(f'CREATE TABLE "{t}" AS SELECT * FROM df')
    con.unregister("df")
con.sql("SELECT table_name, estimated_size AS rows FROM duckdb_tables() ORDER BY 1").show()

┌──────────────────────┬────────┐
│      table_name      │  rows  │
│       varchar        │ int64  │
├──────────────────────┼────────┤
│ circuits             │    150 │
│ constructorResults   │   4597 │
│ constructorStandings │   4950 │
│ constructors         │    425 │
│ driverStandings      │  26741 │
│ drivers              │   1690 │
│ lapTimes             │ 184174 │
│ pitStops             │   2658 │
│ qualifying           │   6700 │
│ races                │    797 │
│ results              │  20212 │
│ seasons              │     68 │
│ status               │    134 │
└──────────────────────┴────────┘
  13 rows             2 columns



In [4]:
def ask(question, sql):
    """Run one AISQL query, show its rows, then what the model calls cost."""
    print(question)
    con.sql(sql).show(max_rows=12)
    usage = con.execute(
        "SELECT llm_calls, cache_hits, input_tokens + output_tokens AS tokens "
        "FROM ai_usage() ORDER BY query_id DESC LIMIT 1").fetchone()
    print(f"model calls {usage[0]}, served from cache {usage[1]}, tokens {usage[2]}\n")

## Questions

The queries are the benchmark's own (`aisql-bench/SWAN2/aisql/` in the SWAN-AISQL repository). Each `ai_filter` prompt keeps the claim apart from the evidence, the layout that measured best on this benchmark.

In [5]:
ask("How many races in the year 2010 were held on circuits in North or South America?  (benchmark answer: 1)", """
WITH circuit_data AS (
    SELECT *, circuits.name || ', ' || circuits.location AS circuit
    FROM circuits
)
SELECT COUNT(DISTINCT races.name || '|' || CAST(races.year AS VARCHAR)) AS races
FROM circuit_data
INNER JOIN races ON races.circuitId = circuit_data.circuitId
WHERE ai_filter('Context:
[circuit]: «' || circuit_data.circuit || '»


Claim: Is this Formula 1 circuit in North or South America? circuit')
  AND races.year = 2010
""")

How many races in the year 2010 were held on circuits in North or South America?  (benchmark answer: 1)


┌───────┐
│ races │
│ int64 │
├───────┤
│     1 │
└───────┘

model calls 14, served from cache 4, tokens 1346



Only the circuits that have a 2010 race reach the model: the relational predicate runs first and the join is semi-reduced before any call, so a handful of prompts decide the count.

In [6]:
ask("Which constructors had a Brazilian driver in a race from 2012 onwards?  (benchmark answer: Ferrari, Sauber, Williams)", """
WITH driver_data AS (
    SELECT *, drivers.forename || ' ' || drivers.surname AS driver
    FROM drivers
)
SELECT DISTINCT constructors.name
FROM results
INNER JOIN races ON races.raceId = results.raceId
INNER JOIN driver_data ON driver_data.driverId = results.driverId
INNER JOIN constructors ON constructors.constructorId = results.constructorId
WHERE races.year >= 2012
  AND ai_filter('Context:
[driver]: «' || driver_data.driver || '»


Claim: Is this Formula 1 driver Brazilian? driver')
""")

Which constructors had a Brazilian driver in a race from 2012 onwards?  (benchmark answer: Ferrari, Sauber, Williams)


┌──────────┐
│   name   │
│ varchar  │
├──────────┤
│ Sauber   │
│ Ferrari  │
│ Williams │
└──────────┘

model calls 55, served from cache 7, tokens 6670



Thousands of result rows, but one call per distinct driver who raced since 2012: the region operator folds the rows into distinct prompts before asking.

In [7]:
ask("Since 2014, which German driver set the highest fastest lap speed in a race, and what was the speed?  (benchmark answer: Sebastian Vettel, 241.626)", """
WITH driver_data AS (
    SELECT *, drivers.forename || ' ' || drivers.surname AS driver
    FROM drivers
)
SELECT driver_data.forename, driver_data.surname, CAST(results.fastestLapSpeed AS DOUBLE) AS speed
FROM results
INNER JOIN races ON races.raceId = results.raceId
INNER JOIN driver_data ON driver_data.driverId = results.driverId
WHERE races.year >= 2014
  AND results.fastestLapSpeed IS NOT NULL
  AND ai_filter('Context:
[driver]: «' || driver_data.driver || '»


Claim: Is this Formula 1 driver German? driver')
ORDER BY speed DESC, driver_data.forename, driver_data.surname
LIMIT 1
""")

Since 2014, which German driver set the highest fastest lap speed in a race, and what was the speed?  (benchmark answer: Sebastian Vettel, 241.626)


┌───────────┬─────────┬─────────┐
│ forename  │ surname │  speed  │
│  varchar  │ varchar │ double  │
├───────────┼─────────┼─────────┤
│ Sebastian │ Vettel  │ 241.626 │
└───────────┴─────────┴─────────┘

model calls 90, served from cache 7, tokens 11711



In [8]:
ask("Name 5 circuits in Europe.", """
SELECT name, location
FROM circuits
WHERE ai_filter('Context:
[circuit]: «' || circuits.name || ', ' || circuits.location || '»


Claim: Is this Formula 1 circuit in Europe? circuit')
LIMIT 5
""")

Name 5 circuits in Europe.


┌────────────────────────────────┬─────────────┐
│              name              │  location   │
│            varchar             │   varchar   │
├────────────────────────────────┼─────────────┤
│ Circuit de Barcelona-Catalunya │ Montmeló    │
│ Circuit de Monaco              │ Monte-Carlo │
│ Circuit de Nevers Magny-Cours  │ Magny Cours │
│ Silverstone Circuit            │ Silverstone │
│ Hockenheimring                 │ Hockenheim  │
└────────────────────────────────┴─────────────┘

model calls 25, served from cache 0, tokens 3829



This one is not a benchmark question but shows the `LIMIT` push-down. The table has 150 rows and 72 distinct circuits; the engine asks about them in waves of twenty and stops as soon as five have passed. The same question with the push-down switched off judges every circuit (`ai_usage()` keeps one row per distinct query text, so the comparison query differs by an alias):

In [9]:
con.execute("SET ai_limit = false")
ask("Name 5 circuits in Europe, judging every circuit first.", """
SELECT name AS circuit, location
FROM circuits
WHERE ai_filter('Context:
[circuit]: «' || circuits.name || ', ' || circuits.location || '»


Claim: Is this Formula 1 circuit in Europe? circuit')
LIMIT 5
""")
con.execute("SET ai_limit = true")

Name 5 circuits in Europe, judging every circuit first.


┌────────────────────────────────┬─────────────┐
│            circuit             │  location   │
│            varchar             │   varchar   │
├────────────────────────────────┼─────────────┤
│ Circuit de Barcelona-Catalunya │ Montmeló    │
│ Circuit de Monaco              │ Monte-Carlo │
│ Circuit de Nevers Magny-Cours  │ Magny Cours │
│ Silverstone Circuit            │ Silverstone │
│ Hockenheimring                 │ Hockenheim  │
└────────────────────────────────┴─────────────┘

model calls 72, served from cache 78, tokens 10980



In [10]:
ask("Which circuits hosted a race in June, July or August both in 1990-1994 and in 1995-1999?  (benchmark answer: Circuit de Nevers Magny-Cours, Circuit Gilles Villeneuve, Hockenheimring, Hungaroring, Silverstone Circuit)", """
WITH race_data AS (
    SELECT *, CAST(races.year AS VARCHAR) || ' ' || races.name AS race
    FROM races
), summer_races AS (
    SELECT race_data.circuitId AS circuitId, race_data.year AS year
    FROM race_data
    WHERE race_data.year BETWEEN 1990 AND 1999
      AND ai_filter('Context:
[race]: «' || race_data.race || '»


Claim: Was this Formula 1 race held in June, July or August? race')
)
SELECT DISTINCT circuits.name
FROM circuits
INNER JOIN summer_races AS early ON early.circuitId = circuits.circuitId
INNER JOIN summer_races AS late ON late.circuitId = circuits.circuitId
WHERE early.year BETWEEN 1990 AND 1994
  AND late.year BETWEEN 1995 AND 1999
""")

Which circuits hosted a race in June, July or August both in 1990-1994 and in 1995-1999?  (benchmark answer: Circuit de Nevers Magny-Cours, Circuit Gilles Villeneuve, Hockenheimring, Hungaroring, Silverstone Circuit)


┌───────────────────────────────┐
│             name              │
│            varchar            │
├───────────────────────────────┤
│ Hungaroring                   │
│ Silverstone Circuit           │
│ Hockenheimring                │
│ Circuit de Nevers Magny-Cours │
│ Circuit Gilles Villeneuve     │
└───────────────────────────────┘

model calls 53, served from cache 73, tokens 6586



The common table expression is read twice; the model is still asked once per race, and the second reference reuses the verdicts.

In [11]:
con.sql("""
SELECT count(*) AS queries, sum(llm_calls) AS model_calls, sum(cache_hits) AS cache_hits,
       sum(input_tokens) AS input_tokens, sum(output_tokens) AS output_tokens
FROM ai_usage() WHERE llm_calls > 0  -- one row per distinct query text
""").show()

┌─────────┬─────────────┬────────────┬──────────────┬───────────────┐
│ queries │ model_calls │ cache_hits │ input_tokens │ output_tokens │
│  int64  │   int128    │   int128   │    int128    │    int128     │
├─────────┼─────────────┼────────────┼──────────────┼───────────────┤
│       6 │         309 │        169 │        27610 │         13512 │
└─────────┴─────────────┴────────────┴──────────────┴───────────────┘



## Further

* Every `SET ai_*` option is a keyword of `connect()`: `connect(ai_concurrency=50)`, `connect(ai_limit=False)` to see a `LIMIT` query judge every row, `connect(embed_server=False)` to skip the embeddings server.
* `ai_classify(text, [labels])`, `ai_score(text, rubric)`, `ai_complete(prompt)`, `ai_agg(list, instruction)` and `ai_image(path)` work the same way; the [README](https://github.com/ZhaoFuheng/SWAN-AISQL) has the full list and the design.
* The other three SWAN 2.0 databases, and the 120 queries, are in the benchmark repository; `SWAN2_DB` selects a different one.